In [7]:
hydrol_diag_soil_string="""
!!
  !& ================================================================================================================================
  !! SUBROUTINE   : hydrol_diag_soil
  !!
  !>\BRIEF        Calculates diagnostic variables at the grid-cell scale
  !!
  !! DESCRIPTION  :
  !! - 1. Apply mask_soiltile
  !! - 2. Sum 3d variables in 2d variables with fraction of vegetation per soil type
  !!
  !! RECENT CHANGE(S) : 2016 by A. Ducharne for the claculation of shumdiag_perma
  !!
  !! MAIN OUTPUT VARIABLE(S) :
  !!
  !! REFERENCE(S) :
  !!
  !! FLOWCHART    : None
  !! \n
  !_
  !& ================================================================================================================================
  !_ hydrol_diag_soil

  SUBROUTINE hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
  !
  ! interface description

  !! 0. Variable and parameter declaration

  !! 0.1 Input variables
  ! input scalar
  INTEGER(KIND = i_std), INTENT(IN) :: kjpindex
  REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(IN) :: veget_max
  !! Max. vegetation type
  INTEGER(KIND = i_std), DIMENSION(kjpindex), INTENT(IN) :: njsc
  !! Index of the dominant soil textural class in the grid cell (1-nscm, unitless)
  REAL(KIND = r_std), DIMENSION(kjpindex, nstm), INTENT(IN) :: soiltile
  !! Fraction of each soil tile within vegtot (0-1, unitless)
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: evapot
  !!
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: returnflow
  !! Water returning to the deep reservoir
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: reinfiltration
  !! Water returning to the top of the soil
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: irrigation
  !! Water from irrigation
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: tot_melt
  !!
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: ks
  !! Hydraulic conductivity at saturation (mm {-1})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: nvan
  !! Van Genuchten coeficients n (unitless)
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: avan
  !! Van Genuchten coeficients a (mm-1})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: mcr
  !! Residual volumetric water content (m^{3} m^{-3})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: mcs
  !! Saturated volumetric water content (m^{3} m^{-3})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: mcfc
  !! Volumetric water content at field capacity (m^{3} m^{-3})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: mcw
  !! Volumetric water content at wilting point (m^{3} m^{-3})
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: precip_rain
  !! Rain precipitation
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: totfrac_nobio
  !! Total fraction of continental ice+lakes+...
  REAL(KIND = r_std), DIMENSION(kjpindex, nnobio), INTENT(IN) :: frac_snow_nobio
  !! Snow cover fraction on non-vegeted area

  !! 0.2 Output variables

  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: drysoil_frac
  !! Function of litter wetness
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: runoff
  !! complete runoff
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: drainage
  !! Drainage
  REAL(KIND = r_std), DIMENSION(kjpindex, nslm), INTENT(OUT) :: shumdiag
  !! relative soil moisture
  REAL(KIND = r_std), DIMENSION(kjpindex, nslm), INTENT(OUT) :: shumdiag_perma
  !! Percent of porosity filled with water (mc/mcs) used for the thermal computations
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: k_litt
  !! litter cond.
  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: litterhumdiag
  !! litter humidity
  REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(OUT) :: humrel
  !! Relative humidity
  REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(OUT) :: vegstress
  !! Veg. moisture stress (only for vegetation growth)

  !! 0.3 Modified variables

  REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(INOUT) :: vevapnu
  !!
  REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm, nslm), INTENT(INOUT) :: us
  !! Water stress index for transpiration
  !! (by soil layer and PFT) (0-1, unitless)


  !! 0.4 Local variables

  INTEGER(KIND = i_std) :: i
  INTEGER(KIND = i_std) :: jst
  INTEGER(KIND = i_std) :: jsl
  INTEGER(KIND = i_std) :: jv
  INTEGER(KIND = i_std) :: ji
  REAL(KIND = r_std), DIMENSION(kjpindex) :: mask_vegtot
  REAL(KIND = r_std) :: tmc_litter_ratio
  REAL(KIND = r_std) :: k_tmp

  !_
  !& ================================================================================================================================
  !
  ! Put the prognostics variables of soil to zero if soiltype is zero

  !! 1. Apply mask_soiltile

  DO jst = 1, nstm
    DO ji = 1, kjpindex

      ae_ns(ji, jst) = ae_ns(ji, jst) * mask_soiltile(ji, jst)
      dr_ns(ji, jst) = dr_ns(ji, jst) * mask_soiltile(ji, jst)
      ru_ns(ji, jst) = ru_ns(ji, jst) * mask_soiltile(ji, jst)
      tmc(ji, jst) = tmc(ji, jst) * mask_soiltile(ji, jst)

        DO jv = 1, nvm
        humrelv(ji, jv, jst) = humrelv(ji, jv, jst) * mask_soiltile(ji, jst)
        DO jsl = 1, nslm
          us(ji, jv, jst, jsl) = us(ji, jv, jst, jsl) * mask_soiltile(ji, jst)
        END DO
      END DO

        DO jsl = 1, nslm
        mc(ji, jsl, jst) = mc(ji, jsl, jst) * mask_soiltile(ji, jst)
      END DO

    END DO
  END DO

  runoff(:) = zero
  drainage(:) = zero
  humtot(:) = zero
  shumdiag(:, :) = zero
  shumdiag_perma(:, :) = zero
  k_litt(:) = zero
  litterhumdiag(:) = zero
  tmc_litt_dry_mea(:) = zero
  tmc_litt_wet_mea(:) = zero
  tmc_litt_mea(:) = zero
  humrel(:, :) = zero
  vegstress(:, :) = zero
  IF (ok_freeze_cwrr) THEN
    profil_froz_hydro(:, :) = zero
    ! initialisation for the mean of profil_froz_hydro_ns
  END IF

    !! 2. Sum 3d variables in 2d variables with fraction of vegetation per soil type

    DO ji = 1, kjpindex
    mask_vegtot(ji) = 0
    IF (vegtot(ji) .GT. min_sechiba) THEN
      mask_vegtot(ji) = 1
    END IF
  END DO

    DO ji = 1, kjpindex
    ! Here we weight ae_ns by the fraction of bare evaporating soil.
    ! This is given by frac_bare_ns, taking into account bare soil under vegetation
    ae_ns(ji, :) = mask_vegtot(ji) * ae_ns(ji, :) * frac_bare_ns(ji, :)
  END DO

    ! We average the values of each soiltile and multiply by vegtot to transform to a grid-cell mean
    DO jst = 1, nstm
    DO ji = 1, kjpindex
      drainage(ji) = mask_vegtot(ji) * (drainage(ji) + vegtot(ji) * soiltile(ji, jst) * dr_ns(ji, jst))
      runoff(ji) = mask_vegtot(ji) * (runoff(ji) + vegtot(ji) * soiltile(ji, jst) * ru_ns(ji, jst)) + (1 - mask_vegtot(ji)) * (tot_melt(ji) + irrigation(ji) + returnflow(ji) + reinfiltration(ji))
      humtot(ji) = mask_vegtot(ji) * (humtot(ji) + vegtot(ji) * soiltile(ji, jst) * tmc(ji, jst))
      IF (ok_freeze_cwrr) THEN
        !  profil_froz_hydro_ns comes from hydrol_soil, to remain the same as in the prognotic loop
        profil_froz_hydro(ji, :) = mask_vegtot(ji) * (profil_froz_hydro(ji, :) + vegtot(ji) * soiltile(ji, jst) * profil_froz_hydro_ns(ji, :, jst))
      END IF
    END DO
  END DO

    ! we add the excess of snow sublimation to vevapnu
    ! - because vevapsno is modified in hydrol_snow if subsinksoil
    ! - it is multiplied by vegtot because it is devided by 1-tot_frac_nobio at creation in hydrol_snow

    DO ji = 1, kjpindex
    IF (vegtot(ji) .NE. 0.) THEN
      vevapnu(ji) = vevapnu(ji) + subsinksoil(ji) * vegtot(ji)
    ELSE
      vevapnu(ji) = vevapnu(ji) + subsinksoil(ji)
    END IF
    runoff(ji) = runoff(ji) + precip_rain(ji) * totfrac_nobio(ji) * (1 - frac_snow_nobio(ji, iice))
  END DO

    DO jst = 1, nstm
    DO jv = 1, nvm
      DO ji = 1, kjpindex
        IF (veget_max(ji, jv) .GT. min_sechiba) THEN
          vegstress(ji, jv) = vegstress(ji, jv) + vegstressv(ji, jv, jst)
          vegstress(ji, jv) = MAX(vegstress(ji, jv), zero)
        END IF
      END DO
    END DO
  END DO

    DO jst = 1, nstm
    DO jv = 1, nvm
      DO ji = 1, kjpindex
        humrel(ji, jv) = humrel(ji, jv) + humrelv(ji, jv, jst)
        humrel(ji, jv) = MAX(humrel(ji, jv), zero)
      END DO
    END DO
  END DO

    !! Litter... the goal is to calculate drysoil_frac, to calculate the albedo in condveg
    ! In condveg, drysoil_frac serve to calculate the albedo of drysoil, excluding the nobio contribution which is further added
    ! In conclusion, we calculate drysoil_frac based on moisture averages restricted to the soiltile (no multiplication by vegtot)
    ! BUT THIS IS NOT USED ANYMORE WITH THE NEW BACKGROUNG ALBEDO
    !! k_litt is calculated here as a grid-cell average (for consistency with drainage)
    !! litterhumdiag, like shumdiag, is averaged over the soiltiles for transmission to stomate
    DO jst = 1, nstm
    DO ji = 1, kjpindex
      ! We compute here a mean k for the 'litter' used for reinfiltration from floodplains of ponds
        IF (tmc_litter(ji, jst) < tmc_litter_res(ji, jst)) THEN
        i = imin
      ELSE
        tmc_litter_ratio = (tmc_litter(ji, jst) - tmc_litter_res(ji, jst)) / (tmc_litter_sat(ji, jst) - tmc_litter_res(ji, jst))
        i = MAX(MIN(INT((imax - imin) * tmc_litter_ratio) + imin, imax - 1), imin)
      END IF
      k_tmp = MAX(k_lin(i, 1, ji) * ks(ji), zero)
      k_litt(ji) = k_litt(ji) + vegtot(ji) * soiltile(ji, jst) * SQRT(k_tmp)
      ! grid-cell average
    END DO
    DO ji = 1, kjpindex
      litterhumdiag(ji) = litterhumdiag(ji) + soil_wet_litter(ji, jst) * soiltile(ji, jst)

      tmc_litt_wet_mea(ji) = tmc_litt_wet_mea(ji) + tmc_litter_awet(ji, jst) * soiltile(ji, jst)

      tmc_litt_dry_mea(ji) = tmc_litt_dry_mea(ji) + tmc_litter_adry(ji, jst) * soiltile(ji, jst)

      tmc_litt_mea(ji) = tmc_litt_mea(ji) + tmc_litter(ji, jst) * soiltile(ji, jst)
    END DO
  END DO

    DO ji = 1, kjpindex
    IF (tmc_litt_wet_mea(ji) - tmc_litt_dry_mea(ji) > zero) THEN
      drysoil_frac(ji) = un + MAX(MIN((tmc_litt_dry_mea(ji) - tmc_litt_mea(ji)) / (tmc_litt_wet_mea(ji) - tmc_litt_dry_mea(ji)), zero), - un)
    ELSE
      drysoil_frac(ji) = zero
    END IF
  END DO

  ! Calculate soilmoist, as a function of total water content (mc)
  ! We average the values of each soiltile and multiply by vegtot to transform to a grid-cell mean
  soilmoist(:, :) = zero
  DO jst = 1, nstm
    DO ji = 1, kjpindex
      soilmoist(ji, 1) = soilmoist(ji, 1) + soiltile(ji, jst) * dz(2) * (trois * mc(ji, 1, jst) + mc(ji, 2, jst)) / huit
      DO jsl = 2, nslm - 1
        soilmoist(ji, jsl) = soilmoist(ji, jsl) + soiltile(ji, jst) * (dz(jsl) * (trois * mc(ji, jsl, jst) + mc(ji, jsl - 1, jst)) / huit + dz(jsl + 1) * (trois * mc(ji, jsl, jst) + mc(ji, jsl + 1, jst)) / huit)
      END DO
      soilmoist(ji, nslm) = soilmoist(ji, nslm) + soiltile(ji, jst) * dz(nslm) * (trois * mc(ji, nslm, jst) + mc(ji, nslm - 1, jst)) / huit
    END DO
  END DO
  DO ji = 1, kjpindex
    soilmoist(ji, :) = soilmoist(ji, :) * vegtot(ji)
    ! conversion to grid-cell average
  END DO

  soilmoist_s(:, :, :) = zero
  DO jst = 1, nstm
    DO ji = 1, kjpindex
      soilmoist_s(ji, 1, nstm) = soilmoist_s(ji, 1, nstm) + soiltile(ji, jst) * dz(2) * (trois * mc(ji, 1, jst) + mc(ji, 2, jst)) / huit
      DO jsl = 2, nslm - 1
        soilmoist_s(ji, jsl, nstm) = soilmoist_s(ji, jsl, nstm) + soiltile(ji, jst) * (dz(jsl) * (trois * mc(ji, jsl, jst) + mc(ji, jsl - 1, jst)) / huit + dz(jsl + 1) * (trois * mc(ji, jsl, jst) + mc(ji, jsl + 1, jst)) / huit)
      END DO
      soilmoist_s(ji, nslm, nstm) = soilmoist_s(ji, nslm, nstm) + soiltile(ji, jst) * dz(nslm) * (trois * mc(ji, nslm, jst) + mc(ji, nslm - 1, jst)) / huit
    END DO
  END DO
  DO ji = 1, kjpindex
    soilmoist_s(ji, :, :) = soilmoist_s(ji, :, :) * vegtot(ji)
    ! conversion to grid-cell average
  END DO

  soilmoist_liquid(:, :) = zero
  DO jst = 1, nstm
    DO ji = 1, kjpindex
      soilmoist_liquid(ji, 1) = soilmoist_liquid(ji, 1) + soiltile(ji, jst) * dz(2) * (trois * mcl(ji, 1, jst) + mcl(ji, 2, jst)) / huit
      DO jsl = 2, nslm - 1
        soilmoist_liquid(ji, jsl) = soilmoist_liquid(ji, jsl) + soiltile(ji, jst) * (dz(jsl) * (trois * mcl(ji, jsl, jst) + mcl(ji, jsl - 1, jst)) / huit + dz(jsl + 1) * (trois * mcl(ji, jsl, jst) + mcl(ji, jsl + 1, jst)) / huit)
      END DO
      soilmoist_liquid(ji, nslm) = soilmoist_liquid(ji, nslm) + soiltile(ji, jst) * dz(nslm) * (trois * mcl(ji, nslm, jst) + mcl(ji, nslm - 1, jst)) / huit
    END DO
  END DO
  DO ji = 1, kjpindex
    soilmoist_liquid(ji, :) = soilmoist_liquid(ji, :) * vegtot_old(ji)
    ! grid cell average
  END DO


    ! Shumdiag: we start from soil_wet_ns, change the range over which the relative moisture is calculated,
    ! then do a spatial average, excluding the nobio fraction on which stomate doesn't act
    DO jst = 1, nstm
    DO jsl = 1, nslm
      DO ji = 1, kjpindex
        shumdiag(ji, jsl) = shumdiag(ji, jsl) + soil_wet_ns(ji, jsl, jst) * soiltile(ji, jst) * ((mcs(ji) - mcw(ji)) / (mcfc(ji) - mcw(ji)))
        shumdiag(ji, jsl) = MAX(MIN(shumdiag(ji, jsl), un), zero)
      END DO
    END DO
  END DO

    ! Shumdiag_perma is based on soilmoist / moisture at saturation in the layer
    ! Her we start from grid averages by hydrol soil layer and transform it to the diag levels
    ! We keep a grid-cell average, like for all variables transmitted to ok_freeze
    DO jsl = 1, nslm
    DO ji = 1, kjpindex
      shumdiag_perma(ji, jsl) = soilmoist(ji, jsl) / (dh(jsl) * mcs(ji))
      shumdiag_perma(ji, jsl) = MAX(MIN(shumdiag_perma(ji, jsl), un), zero)
    END DO
  END DO

END SUBROUTINE hydrol_diag_soil
"""

In [4]:
# change to the Fgpt directory
%cd ../

/home/kardaneh/Fgpt


/home/kardaneh/fparser_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
from processor import Processor
from fparser.two import Fortran2003 as F23
processor = Processor()

INFO     Processor initialized.

In [8]:
tree = processor.parse_fortran_string(hydrol_diag_soil_string)
tree

INFO     Successfully parsed string!

Program(Comment(''), Comment('!!'), Comment('!& ================================================================================================================================'), Comment('!! SUBROUTINE   : hydrol_diag_soil'), Comment('!!'), Comment('!>\\BRIEF        Calculates diagnostic variables at the grid-cell scale'), Comment('!!'), Comment('!! DESCRIPTION  :'), Comment('!! - 1. Apply mask_soiltile'), Comment('!! - 2. Sum 3d variables in 2d variables with fraction of vegetation per soil type'), Comment('!!'), Comment('!! RECENT CHANGE(S) : 2016 by A. Ducharne for the claculation of shumdiag_perma'), Comment('!!'), Comment('!! MAIN OUTPUT VARIABLE(S) :'), Comment('!!'), Comment('!! REFERENCE(S) :'), Comment('!!'), Comment('!! FLOWCHART    : None'), Comment('!!'), Comment(''), Comment('!_'), Comment('!& ================================================================================================================================'), Comment('!_ hydrol_diag_soil'), Comment(''), Subr

In [10]:
from f2np import F2NP

f2np_ = F2NP()

In [11]:
f2np_.recursive(tree)

SUBROUTINE hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
def hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio):

INTEGER(KIND = i_std), INTENT(IN) :: kjpindex
None

REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(IN) :: veget_max
None

INTEGER(KIND = i_std), DIMENSION(kjpindex), INTENT(IN) :: njsc
None

REAL(KIND = r_std), DIMENSION(kjpindex, nstm), INTENT(IN) :: soiltile
None

REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: evapot
None

REAL(KIND = r_std), DIMENSIO

In [7]:
f2np_.intrinsic_replacements

{'\\bINT\\b': 'int',
 '\\bMIN\\b': 'min',
 '\\bMAX\\b': 'max',
 '\\bMAXVAL\\b': 'np.max',
 '\\bMINVAL\\b': 'np.min',
 '\\bABS\\b': 'np.abs',
 '\\bSQRT\\b': 'np.sqrt',
 '\\bEXP\\b': 'np.exp',
 '\\bLOG\\b': 'np.log',
 '\\bSIN\\b': 'np.sin',
 '\\bCOS\\b': 'np.cos',
 '\\bTAN\\b': 'np.tan',
 '\\bASIN\\b': 'np.arcsin',
 '\\bACOS\\b': 'np.arccos',
 '\\bATAN\\b': 'np.arctan',
 '\\bATAN2\\b': 'np.arctan2',
 '\\bMOD\\b': 'np.mod',
 '\\bCEILING\\b': 'np.ceil',
 '\\bFLOOR\\b': 'np.floor',
 '\\bSUM\\b': 'np.sum',
 '\\bPRODUCT\\b': 'np.prod',
 '\\bDOT_PRODUCT\\b': 'np.dot',
 '\\bMATMUL\\b': 'np.matmul',
 '\\bRESHAPE\\b': 'np.reshape',
 '\\bALLOCATE\\b': 'np.empty',
 '\\bSIZE\\b': 'np.size'}

In [12]:
loop_node= F23.Nonlabel_Do_Stmt("DO jsl = 0, nslm")

In [13]:
loop_node

Nonlabel_Do_Stmt('DO', Loop_Control(None, (Name('jsl'), [Int_Literal_Constant('0', None), Name('nslm')]), None))

In [14]:
f2np_.handle_do_stmt(loop_node)

'for jsl in range(- 1, nslm, 1):'